# MÔ HÌNH HydraSwin

# BƯỚC 1 — Import thư viện

In [ ]:
!pip install timm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import timm
import pandas as pd


# BƯỚC 2 — Load dataset

In [37]:
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.preprocessing import LabelEncoder
import torch
import os

# 1. Đọc file CSV
df = pd.read_csv(
    "/kaggle/input/datasets/ndidang/deeplearningndidangdatasets/styles.csv",
    engine="python",
    on_bad_lines="skip"
)

# 2. Lọc bỏ các hàng có NaN trong 6 cột nhãn chính
df = df.dropna(subset=['gender', 'masterCategory', 'subCategory', 
                       'articleType', 'baseColour', 'season']).reset_index(drop=True)

# 3. Lọc bỏ những hàng không có ảnh tồn tại (tối ưu bằng os.listdir)
image_dir = "/kaggle/input/datasets/ndidang/deeplearningndidangdatasets/images"
available_images = set([f.split('.')[0] for f in os.listdir(image_dir)])
df = df[df['id'].astype(str).isin(available_images)].reset_index(drop=True)

print("✅ Dataset sau khi lọc NaN và lọc ảnh tồn tại:", df.shape)

# 4. Tạo encoder cho từng cột
gender_encoder = LabelEncoder().fit(df['gender'])
master_encoder = LabelEncoder().fit(df['masterCategory'])
sub_encoder = LabelEncoder().fit(df['subCategory'])
article_encoder = LabelEncoder().fit(df['articleType'])
colour_encoder = LabelEncoder().fit(df['baseColour'])
season_encoder = LabelEncoder().fit(df['season'])

# 5. Chuẩn hóa ảnh
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

# 6. Dataset class
class FashionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(image_dir, str(row['id']) + ".jpg")
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        labels = {
            "gender": torch.tensor(gender_encoder.transform([row['gender']])[0], dtype=torch.long),
            "master_cat": torch.tensor(master_encoder.transform([row['masterCategory']])[0], dtype=torch.long),
            "sub_cat": torch.tensor(sub_encoder.transform([row['subCategory']])[0], dtype=torch.long),
            "article_type": torch.tensor(article_encoder.transform([row['articleType']])[0], dtype=torch.long),
            "base_colour": torch.tensor(colour_encoder.transform([row['baseColour']])[0], dtype=torch.long),
            "season": torch.tensor(season_encoder.transform([row['season']])[0], dtype=torch.long)
        }
        return image, labels

# 7. DataLoader
train_dataset = FashionDataset(df, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


✅ Dataset sau khi lọc NaN và lọc ảnh tồn tại: (44383, 10)


In [38]:
# Đếm số lớp cho từng task theo tên cột gốc trong styles.csv
num_classes_dict = {
    "gender": df['gender'].nunique(),
    "master_cat": df['masterCategory'].nunique(),
    "sub_cat": df['subCategory'].nunique(),
    "article_type": df['articleType'].nunique(),
    "base_colour": df['baseColour'].nunique(),
    "season": df['season'].nunique()
}

print(num_classes_dict)


{'gender': 5, 'master_cat': 7, 'sub_cat': 45, 'article_type': 142, 'base_colour': 46, 'season': 4}


# BƯỚC 3 — Mô hình HydraSwin (Swin backbone + 6 heads + Top-K)

In [39]:
import torch.nn as nn
import timm

class HydraSwin(nn.Module):
    def __init__(self, num_classes_dict):
        super(HydraSwin, self).__init__()
        # Backbone Swin Transformer (Tiny)
        self.backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True)
        self.backbone.reset_classifier(0)  # bỏ head gốc
        feat_dim = 768  # Swin Tiny output dimension

        # 6 heads cho 6 tasks
        self.gender_head = nn.Linear(feat_dim, num_classes_dict['gender'])
        self.master_cat_head = nn.Linear(feat_dim, num_classes_dict['master_cat'])
        self.sub_cat_head = nn.Linear(feat_dim, num_classes_dict['sub_cat'])
        self.article_type_head = nn.Linear(feat_dim, num_classes_dict['article_type'])
        self.base_colour_head = nn.Linear(feat_dim, num_classes_dict['base_colour'])
        self.season_head = nn.Linear(feat_dim, num_classes_dict['season'])

    def forward(self, x, topk=1):
        features = self.backbone(x)
        outputs = {
            "gender": self.gender_head(features),
            "master_cat": self.master_cat_head(features),
            "sub_cat": self.sub_cat_head(features),
            "article_type": self.article_type_head(features),
            "base_colour": self.base_colour_head(features),
            "season": self.season_head(features)
        }
        # Top-K strategy để giả lập multi-label
        preds_topk = {task: torch.topk(logits, k=topk, dim=1) for task, logits in outputs.items()}
        return outputs, preds_topk


# BƯỚC 4 — Weighted Loss cho imbalance

In [40]:
import torch.nn as nn
import torch
from sklearn.preprocessing import LabelEncoder

# Hàm tính trọng số cho từng cột
def compute_class_weights(series):
    # Bỏ NaN trước khi tính
    series = series.dropna()
    counts = series.value_counts()
    total = len(series)
    weights = {cls: total/(count+1e-6) for cls, count in counts.items()}
    encoder = LabelEncoder().fit(series)
    weight_list = [weights[cls] for cls in encoder.classes_]
    return weight_list


# Tính trọng số cho từng task
class_weights_dict = {
    "gender": compute_class_weights(df['gender']),
    "master_cat": compute_class_weights(df['masterCategory']),
    "sub_cat": compute_class_weights(df['subCategory']),
    "article_type": compute_class_weights(df['articleType']),
    "base_colour": compute_class_weights(df['baseColour']),
    "season": compute_class_weights(df['season'])
}

# Hàm Weighted Loss
def get_weighted_loss(outputs, labels, class_weights_dict):
    total_loss = 0
    for task in outputs.keys():
        weight = class_weights_dict.get(task, None)
        if weight is not None:
            weight = torch.tensor(weight).float().to(outputs[task].device)
        criterion = nn.CrossEntropyLoss(weight=weight)
        total_loss += criterion(outputs[task], labels[task])
    return total_loss


# BƯỚC 5 — Training loop + Lưu model

In [41]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Số lớp cho từng task
num_classes_dict = {
    "gender": df['gender'].nunique(),
    "master_cat": df['masterCategory'].nunique(),
    "sub_cat": df['subCategory'].nunique(),
    "article_type": df['articleType'].nunique(),
    "base_colour": df['baseColour'].nunique(),
    "season": df['season'].nunique()
}

# Khởi tạo model HydraSwin
model = HydraSwin(num_classes_dict).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training loop
num_epochs = 30  
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = {k: v.to(device) for k,v in labels.items()}

        outputs, _ = model(images)
        loss = get_weighted_loss(outputs, labels, class_weights_dict)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")

# Lưu model
torch.save(model.state_dict(), "/kaggle/working/hydraswin_model.pth")
print("✅ Model đã được lưu tại /kaggle/working/hydraswin_model.pth")


Epoch 1/30, Loss: 8.5109
Epoch 2/30, Loss: 4.8891
Epoch 3/30, Loss: 3.6636
Epoch 4/30, Loss: 2.9537
Epoch 5/30, Loss: 2.4844
Epoch 6/30, Loss: 2.0889
Epoch 7/30, Loss: 1.8080
Epoch 8/30, Loss: 1.5509
Epoch 9/30, Loss: 1.4905
Epoch 10/30, Loss: 1.4071
Epoch 11/30, Loss: 1.1180
Epoch 12/30, Loss: 0.9874
Epoch 13/30, Loss: 0.8880
Epoch 14/30, Loss: 0.8365
Epoch 15/30, Loss: 0.8066
Epoch 16/30, Loss: 0.6426
Epoch 17/30, Loss: 0.7659
Epoch 18/30, Loss: 0.7732
Epoch 19/30, Loss: 0.7295
Epoch 20/30, Loss: 0.6474
Epoch 21/30, Loss: 0.5285
Epoch 22/30, Loss: 0.4481
Epoch 23/30, Loss: 0.4468
Epoch 24/30, Loss: 0.6592
Epoch 25/30, Loss: 0.4649
Epoch 26/30, Loss: 0.3675
Epoch 27/30, Loss: 0.4433
Epoch 28/30, Loss: 0.4843
Epoch 29/30, Loss: 0.3670
Epoch 30/30, Loss: 0.5433
✅ Model đã được lưu tại /kaggle/working/hydraswin_model.pth


# BƯỚC 6 — Evaluation (đánh giá mô hình)

In [42]:
from sklearn.metrics import f1_score

model.eval()
all_preds, all_labels = {task: [] for task in num_classes_dict}, {task: [] for task in num_classes_dict}

with torch.no_grad():
    for images, labels in train_loader:  # hoặc val_loader nếu bạn có tập validation
        images = images.to(device)
        labels = {k: v.to(device) for k,v in labels.items()}
        outputs, _ = model(images)

        for task in outputs.keys():
            preds = torch.argmax(outputs[task], dim=1)
            all_preds[task].extend(preds.cpu().numpy())
            all_labels[task].extend(labels[task].cpu().numpy())

for task in all_preds.keys():
    f1 = f1_score(all_labels[task], all_preds[task], average="macro")
    print(f"{task} Macro F1: {f1:.4f}")


gender Macro F1: 0.9828
master_cat Macro F1: 0.9857
sub_cat Macro F1: 0.9784
article_type Macro F1: 0.9647
base_colour Macro F1: 0.8630
season Macro F1: 0.9852


# BƯỚC 7 — Encode label

In [43]:
from sklearn.preprocessing import LabelEncoder

gender_encoder = LabelEncoder().fit(df['gender'])
master_encoder = LabelEncoder().fit(df['masterCategory'])
sub_encoder = LabelEncoder().fit(df['subCategory'])
article_encoder = LabelEncoder().fit(df['articleType'])
colour_encoder = LabelEncoder().fit(df['baseColour'])
season_encoder = LabelEncoder().fit(df['season'])


# BƯỚC 8 — Train Validation split

In [44]:
import joblib

joblib.dump(gender_encoder, "/kaggle/working/gender_encoder.pkl")
joblib.dump(master_encoder, "/kaggle/working/master_encoder.pkl")
joblib.dump(sub_encoder, "/kaggle/working/sub_encoder.pkl")
joblib.dump(article_encoder, "/kaggle/working/article_encoder.pkl")
joblib.dump(colour_encoder, "/kaggle/working/colour_encoder.pkl")
joblib.dump(season_encoder, "/kaggle/working/season_encoder.pkl")


['/kaggle/working/season_encoder.pkl']

# Mô hình HydraMultiTask

# BƯỚC 1 — Import thư viện

In [1]:
!pip install timm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import timm
import pandas as pd

# BƯỚC 2 — Load dataset

In [2]:
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.preprocessing import LabelEncoder
import torch
import os

# 1. Đọc file CSV
df = pd.read_csv(
    "/kaggle/input/datasets/ndidang/deeplearningndidangdatasets/styles.csv",
    engine="python",
    on_bad_lines="skip"
)

# 2. Lọc bỏ các hàng có NaN trong 6 cột nhãn chính
df = df.dropna(subset=['gender', 'masterCategory', 'subCategory', 
                       'articleType', 'baseColour', 'season']).reset_index(drop=True)

# 3. Lọc bỏ những hàng không có ảnh tồn tại (tối ưu bằng os.listdir)
image_dir = "/kaggle/input/datasets/ndidang/deeplearningndidangdatasets/images"
available_images = set([f.split('.')[0] for f in os.listdir(image_dir)])
df = df[df['id'].astype(str).isin(available_images)].reset_index(drop=True)

print("✅ Dataset sau khi lọc NaN và lọc ảnh tồn tại:", df.shape)

# 4. Tạo encoder cho từng cột
gender_encoder = LabelEncoder().fit(df['gender'])
master_encoder = LabelEncoder().fit(df['masterCategory'])
sub_encoder = LabelEncoder().fit(df['subCategory'])
article_encoder = LabelEncoder().fit(df['articleType'])
colour_encoder = LabelEncoder().fit(df['baseColour'])
season_encoder = LabelEncoder().fit(df['season'])

# 5. Chuẩn hóa ảnh
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

# 6. Dataset class
class FashionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(image_dir, str(row['id']) + ".jpg")
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        labels = {
            "gender": torch.tensor(gender_encoder.transform([row['gender']])[0], dtype=torch.long),
            "master_cat": torch.tensor(master_encoder.transform([row['masterCategory']])[0], dtype=torch.long),
            "sub_cat": torch.tensor(sub_encoder.transform([row['subCategory']])[0], dtype=torch.long),
            "article_type": torch.tensor(article_encoder.transform([row['articleType']])[0], dtype=torch.long),
            "base_colour": torch.tensor(colour_encoder.transform([row['baseColour']])[0], dtype=torch.long),
            "season": torch.tensor(season_encoder.transform([row['season']])[0], dtype=torch.long)
        }
        return image, labels

# 7. DataLoader
train_dataset = FashionDataset(df, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


✅ Dataset sau khi lọc NaN và lọc ảnh tồn tại: (44383, 10)


In [3]:
# Đếm số lớp cho từng task theo tên cột gốc trong styles.csv
num_classes_dict = {
    "gender": df['gender'].nunique(),
    "master_cat": df['masterCategory'].nunique(),
    "sub_cat": df['subCategory'].nunique(),
    "article_type": df['articleType'].nunique(),
    "base_colour": df['baseColour'].nunique(),
    "season": df['season'].nunique()
}

print(num_classes_dict)


{'gender': 5, 'master_cat': 7, 'sub_cat': 45, 'article_type': 142, 'base_colour': 46, 'season': 4}


# BƯỚC 3 — Mô hình HydraMultiTask (Tích hợp tự động SOTA Baselines)

In [7]:
import torch
import torch.nn as nn
import timm

class HydraMultiTaskModel(nn.Module):
    def __init__(self, num_classes_dict, model_name="swin_tiny_patch4_window7_224"):
        super(HydraMultiTaskModel, self).__init__()
        
        # Tạo backbone tự động và loại bỏ classifier gốc bằng tham số num_classes=0
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        
        # Tự động lấy kích thước đầu ra bằng cách cho chạy thử 1 tensor ảo qua backbone
        # Cách này tuyệt đối chính xác 100% cho mọi loại model (Swin, ViT, ConvNeXt, MobileNet)
        dummy_x = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            dummy_features = self.backbone(dummy_x)
        feat_dim = dummy_features.shape[1] 
        
        print(f"✅ Backbone {model_name} initialized with feature dimension: {feat_dim}")
        
        # 6 classification heads cho 6 tasks kết nối trực tiếp với backbone
        self.gender_head = nn.Linear(feat_dim, num_classes_dict['gender'])
        self.master_cat_head = nn.Linear(feat_dim, num_classes_dict['master_cat'])
        self.sub_cat_head = nn.Linear(feat_dim, num_classes_dict['sub_cat'])
        self.article_type_head = nn.Linear(feat_dim, num_classes_dict['article_type'])
        self.base_colour_head = nn.Linear(feat_dim, num_classes_dict['base_colour'])
        self.season_head = nn.Linear(feat_dim, num_classes_dict['season'])

    def forward(self, x, topk=1):
        features = self.backbone(x)
        outputs = {
            "gender": self.gender_head(features),
            "master_cat": self.master_cat_head(features),
            "sub_cat": self.sub_cat_head(features),
            "article_type": self.article_type_head(features),
            "base_colour": self.base_colour_head(features),
            "season": self.season_head(features)
        }
        # Top-K strategy để lọc logic rules sau này
        preds_topk = {task: torch.topk(logits, k=topk, dim=1) for task, logits in outputs.items()}
        return outputs, preds_topk

# BƯỚC 4 — Weighted Loss cho imbalance

In [8]:
import torch.nn as nn
import torch
from sklearn.preprocessing import LabelEncoder

# Hàm tính trọng số cho từng cột
def compute_class_weights(series):
    # Bỏ NaN trước khi tính
    series = series.dropna()
    counts = series.value_counts()
    total = len(series)
    weights = {cls: total/(count+1e-6) for cls, count in counts.items()}
    encoder = LabelEncoder().fit(series)
    weight_list = [weights[cls] for cls in encoder.classes_]
    return weight_list


# Tính trọng số cho từng task
class_weights_dict = {
    "gender": compute_class_weights(df['gender']),
    "master_cat": compute_class_weights(df['masterCategory']),
    "sub_cat": compute_class_weights(df['subCategory']),
    "article_type": compute_class_weights(df['articleType']),
    "base_colour": compute_class_weights(df['baseColour']),
    "season": compute_class_weights(df['season'])
}

# Hàm Weighted Loss
def get_weighted_loss(outputs, labels, class_weights_dict):
    total_loss = 0
    for task in outputs.keys():
        weight = class_weights_dict.get(task, None)
        if weight is not None:
            weight = torch.tensor(weight).float().to(outputs[task].device)
        criterion = nn.CrossEntropyLoss(weight=weight)
        total_loss += criterion(outputs[task], labels[task])
    return total_loss


# BƯỚC 5 — Training loop + Lưu model (Hỗ trợ cấu hình chọn Model)


In [9]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# =====================================================================
# DANH SÁCH SOTA MODELS (Chỉ cần đổi tên biến TARGET_MODEL ở dưới)
# 1. 'swin_tiny_patch4_window7_224' (Mô hình gốc HydraSwin của bài)
# 2. 'vit_tiny_patch16_224'         (Baseline: Vision Transformer chuẩn)
# 3. 'convnext_tiny'                (Baseline: CNN hiện đại mạnh nhất)
# 4. 'mobilenetv3_large_100'        (Baseline: Nhẹ, tối ưu độ trễ)
# =====================================================================
TARGET_MODEL = "mobilenetv3_large_100" 
print(f"🚀 Bắt đầu huấn luyện với kiến trúc: {TARGET_MODEL}")
# Khởi tạo model đa nhiệm tự động khớp kiến trúc
model = HydraMultiTaskModel(num_classes_dict, model_name=TARGET_MODEL).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# Training loop
num_epochs = 30  
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = {k: v.to(device) for k,v in labels.items()}
        outputs, _ = model(images)
        loss = get_weighted_loss(outputs, labels, class_weights_dict)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")
# Lưu model với tên linh hoạt để không bị ghi đè
save_path = f"/kaggle/working/hydra_{TARGET_MODEL}.pth"
torch.save(model.state_dict(), save_path)
print(f"✅ Model đã được lưu tại {save_path}")

🚀 Bắt đầu huấn luyện với kiến trúc: mobilenetv3_large_100
✅ Backbone mobilenetv3_large_100 initialized with feature dimension: 1280
Epoch 1/30, Loss: 9.0207
Epoch 2/30, Loss: 4.8144
Epoch 3/30, Loss: 3.5913
Epoch 4/30, Loss: 2.8677
Epoch 5/30, Loss: 2.4224
Epoch 6/30, Loss: 2.1540
Epoch 7/30, Loss: 1.8451
Epoch 8/30, Loss: 1.6557
Epoch 9/30, Loss: 1.4617
Epoch 10/30, Loss: 1.2696
Epoch 11/30, Loss: 1.1513
Epoch 12/30, Loss: 1.0370
Epoch 13/30, Loss: 0.9100
Epoch 14/30, Loss: 0.8106
Epoch 15/30, Loss: 0.7983
Epoch 16/30, Loss: 0.7068
Epoch 17/30, Loss: 0.6062
Epoch 18/30, Loss: 0.5617
Epoch 19/30, Loss: 0.4907
Epoch 20/30, Loss: 0.5065
Epoch 21/30, Loss: 0.5739
Epoch 22/30, Loss: 0.4143
Epoch 23/30, Loss: 0.3755
Epoch 24/30, Loss: 0.3041
Epoch 25/30, Loss: 0.3199
Epoch 26/30, Loss: 0.3100
Epoch 27/30, Loss: 0.3011
Epoch 28/30, Loss: 0.2885
Epoch 29/30, Loss: 0.3571
Epoch 30/30, Loss: 0.2977
✅ Model đã được lưu tại /kaggle/working/hydra_mobilenetv3_large_100.pth


# Đánh giá : Hydra Vit Tiny

In [9]:
from sklearn.metrics import f1_score

model.eval()
all_preds, all_labels = {task: [] for task in num_classes_dict}, {task: [] for task in num_classes_dict}

with torch.no_grad():
    for images, labels in train_loader:  # hoặc val_loader nếu bạn có tập validation
        images = images.to(device)
        labels = {k: v.to(device) for k,v in labels.items()}
        outputs, _ = model(images)

        for task in outputs.keys():
            preds = torch.argmax(outputs[task], dim=1)
            all_preds[task].extend(preds.cpu().numpy())
            all_labels[task].extend(labels[task].cpu().numpy())

for task in all_preds.keys():
    f1 = f1_score(all_labels[task], all_preds[task], average="macro")
    print(f"{task} Macro F1: {f1:.4f}")


gender Macro F1: 0.9611
master_cat Macro F1: 0.9795
sub_cat Macro F1: 0.9679
article_type Macro F1: 0.9540
base_colour Macro F1: 0.8366
season Macro F1: 0.9474


# Đánh giá : Hydra Convnext Tiny

In [14]:
from sklearn.metrics import f1_score

model.eval()
all_preds, all_labels = {task: [] for task in num_classes_dict}, {task: [] for task in num_classes_dict}

with torch.no_grad():
    for images, labels in train_loader:  # hoặc val_loader nếu bạn có tập validation
        images = images.to(device)
        labels = {k: v.to(device) for k,v in labels.items()}
        outputs, _ = model(images)

        for task in outputs.keys():
            preds = torch.argmax(outputs[task], dim=1)
            all_preds[task].extend(preds.cpu().numpy())
            all_labels[task].extend(labels[task].cpu().numpy())

for task in all_preds.keys():
    f1 = f1_score(all_labels[task], all_preds[task], average="macro")
    print(f"{task} Macro F1: {f1:.4f}")

gender Macro F1: 0.9937
master_cat Macro F1: 0.9784
sub_cat Macro F1: 0.9962
article_type Macro F1: 0.9922
base_colour Macro F1: 0.9275
season Macro F1: 0.9937


# Đánh giá : Hydra mobilenetv3_large_100

In [10]:
from sklearn.metrics import f1_score

model.eval()
all_preds, all_labels = {task: [] for task in num_classes_dict}, {task: [] for task in num_classes_dict}

with torch.no_grad():
    for images, labels in train_loader:  # hoặc val_loader nếu bạn có tập validation
        images = images.to(device)
        labels = {k: v.to(device) for k,v in labels.items()}
        outputs, _ = model(images)

        for task in outputs.keys():
            preds = torch.argmax(outputs[task], dim=1)
            all_preds[task].extend(preds.cpu().numpy())
            all_labels[task].extend(labels[task].cpu().numpy())

for task in all_preds.keys():
    f1 = f1_score(all_labels[task], all_preds[task], average="macro")
    print(f"{task} Macro F1: {f1:.4f}")

gender Macro F1: 0.9940
master_cat Macro F1: 0.9976
sub_cat Macro F1: 0.9942
article_type Macro F1: 0.9881
base_colour Macro F1: 0.9055
season Macro F1: 0.9938


# BƯỚC 7 — Encode label

In [11]:
from sklearn.preprocessing import LabelEncoder

gender_encoder = LabelEncoder().fit(df['gender'])
master_encoder = LabelEncoder().fit(df['masterCategory'])
sub_encoder = LabelEncoder().fit(df['subCategory'])
article_encoder = LabelEncoder().fit(df['articleType'])
colour_encoder = LabelEncoder().fit(df['baseColour'])
season_encoder = LabelEncoder().fit(df['season'])


# BƯỚC 8 — Train Validation split

In [12]:
import joblib

joblib.dump(gender_encoder, "/kaggle/working/gender_encoder.pkl")
joblib.dump(master_encoder, "/kaggle/working/master_encoder.pkl")
joblib.dump(sub_encoder, "/kaggle/working/sub_encoder.pkl")
joblib.dump(article_encoder, "/kaggle/working/article_encoder.pkl")
joblib.dump(colour_encoder, "/kaggle/working/colour_encoder.pkl")
joblib.dump(season_encoder, "/kaggle/working/season_encoder.pkl")


['/kaggle/working/season_encoder.pkl']